# BBCDS research baseline training

This notebook trains the BBCDS MobileNetV3-Small research baseline on a
sensitive, gated dataset. Run it only if you are an adult, access is lawful
where you live, and you have accepted the dataset's terms.

The dataset card does not establish provenance, consent, or training rights
for every underlying image. The result is not commercially cleared.

Raw images and the protected manifest stay in the temporary Colab runtime.
Google Drive stores checkpoints and aggregate model outputs only.

## 1. Confirm the GPU runtime

In Colab, select **Runtime > Change runtime type > T4 GPU** before running
this cell. Free GPU access is limited and can be interrupted.

In [ ]:
from __future__ import annotations

import subprocess

gpu_status = subprocess.run(
    ["nvidia-smi"],
    check=False,
    capture_output=True,
    text=True,
)
if gpu_status.returncode != 0:
    raise RuntimeError("No GPU is attached. Select a GPU runtime and reconnect.")
print(gpu_status.stdout.splitlines()[0])

## 2. Install uv

In [ ]:
%pip install -q uv

## 3. Download the current repository

Public repositories need no GitHub credential. For a private repository, add
a read-only `GITHUB_TOKEN` in the Colab Secrets panel and enable notebook
access to it.

In [ ]:
import base64
import os
from pathlib import Path

from google.colab import userdata

REPOSITORY_URL = "https://github.com/oddegen/bbcds.git"
REPOSITORY_ROOT = Path("/content/bbcds")


def optional_secret(name: str) -> str | None:
    try:
        return userdata.get(name)
    except userdata.SecretNotFoundError:
        return None


git_env = os.environ.copy()
github_token = optional_secret("GITHUB_TOKEN")
if github_token:
    credentials = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_env.update(
        {
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.extraHeader",
            "GIT_CONFIG_VALUE_0": f"Authorization: Basic {credentials}",
        }
    )

if REPOSITORY_ROOT.exists():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPOSITORY_ROOT, env=git_env, check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_ROOT)], env=git_env, check=True)

## 4. Create the pinned model environment

In [ ]:
MODEL_ROOT = REPOSITORY_ROOT / "model"
subprocess.run(["uv", "sync"], cwd=MODEL_ROOT, check=True)
PYTHON = MODEL_ROOT / ".venv" / "bin" / "python"

## 5. Download and extract the gated research dataset

First accept the access conditions on the dataset page. Then create a
read-only Hugging Face token, save it as `HF_TOKEN` in Colab Secrets, and
enable notebook access to it.

In [ ]:
import zipfile

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is missing from Colab Secrets.")

DATA_ROOT = Path("/content/bbcds-data")
EXTRACT_ROOT = DATA_ROOT / "extracted"
MANIFEST_PATH = DATA_ROOT / "dataset.csv"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

download_env = os.environ.copy()
download_env["HF_TOKEN"] = hf_token
download = subprocess.run(
    [
        str(MODEL_ROOT / ".venv" / "bin" / "hf"),
        "download",
        "deepghs/nsfw_detect",
        "nsfw_dataset_v1.zip",
        "--repo-type",
        "dataset",
        "--local-dir",
        str(DATA_ROOT),
    ],
    env=download_env,
    check=True,
    capture_output=True,
    text=True,
)
archive_path = Path(download.stdout.strip().splitlines()[-1])
if not EXTRACT_ROOT.exists():
    with zipfile.ZipFile(archive_path) as archive:
        extraction_base = EXTRACT_ROOT.resolve()
        for member in archive.infolist():
            member_path = (EXTRACT_ROOT / member.filename).resolve()
            if not member_path.is_relative_to(extraction_base):
                raise RuntimeError("The dataset archive contains an unsafe path.")
        archive.extractall(EXTRACT_ROOT)

required_folders = {"drawing", "neutral", "sexy", "porn", "hentai"}
dataset_roots = [
    path
    for path in [EXTRACT_ROOT, *EXTRACT_ROOT.iterdir()]
    if path.is_dir() and required_folders.issubset({child.name.lower() for child in path.iterdir()})
]
if len(dataset_roots) != 1:
    raise RuntimeError("The downloaded archive does not have the expected folder structure.")
DATASET_ROOT = dataset_roots[0]
print("Dataset downloaded to temporary Colab storage.")

## 6. Build and validate the protected manifest

The command verifies images, hashes them, removes exact duplicates, groups
near-duplicates, excludes conflicting-label groups, and creates deterministic
source-grouped train, validation, and test splits.

In [ ]:
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "bbcds_model.prepare_manifest",
        "--dataset-root",
        str(DATASET_ROOT),
        "--output",
        str(MANIFEST_PATH),
        "--profile",
        "deepghs-nsfw-detect",
        "--seed",
        "20260731",
    ],
    cwd=MODEL_ROOT,
    check=True,
)

## 7. Review aggregate preparation counts

This cell intentionally displays no filenames, paths, image data, or model
probabilities.

In [ ]:
import json

audit = json.loads(MANIFEST_PATH.with_suffix(".audit.json").read_text())
for key in (
    "scannedCount",
    "acceptedCount",
    "corruptCount",
    "policyExcludedCount",
    "exactDuplicateCount",
    "conflictingClusterRecordCount",
    "nearDuplicateClusterCount",
    "sourceGroupCount",
    "labels",
    "splits",
):
    print(f"{key}: {audit[key]}")

## 8. Verify TensorFlow can use the GPU

In [ ]:
gpu_check = subprocess.run(
    [
        str(PYTHON),
        "-c",
        (
            "import tensorflow as tf; "
            "gpus=tf.config.list_physical_devices('GPU'); "
            "assert gpus, 'TensorFlow cannot see a GPU'; "
            "print('TensorFlow GPUs:', len(gpus))"
        ),
    ],
    cwd=MODEL_ROOT,
    check=False,
    capture_output=True,
    text=True,
)
if gpu_check.returncode != 0:
    if gpu_check.stdout:
        print(gpu_check.stdout)
    if gpu_check.stderr:
        print(gpu_check.stderr)
    raise RuntimeError("TensorFlow GPU check failed.")

print(gpu_check.stdout.strip())

## 9. Mount Drive for checkpoints and outputs

Do not move the dataset or protected manifest into Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR = Path("/content/drive/MyDrive/bbcds-runs/mobilenet-v3-small-v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 10. Train or recover the model

Rerunning this cell with the same Drive folder continues from the newest
valid checkpoint. If Colab reports GPU memory exhaustion, change `64` to `32`.

In [ ]:
commit = subprocess.run(
    ["git", "rev-parse", "--short=12", "HEAD"],
    cwd=REPOSITORY_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

training_command = [
    str(PYTHON),
    "-m",
    "bbcds_model.train",
    "--manifest",
    str(MANIFEST_PATH),
    "--output-dir",
    str(OUTPUT_DIR),
    "--batch-size",
    "64",
    "--training-commit",
    commit,
    "--resume",
]
subprocess.run(training_command, cwd=MODEL_ROOT, check=True)

## 11. Verify preserved outputs

In [ ]:
required_outputs = [
    OUTPUT_DIR / "final.keras",
    OUTPUT_DIR / "run-metadata.json",
    OUTPUT_DIR / "baseline-validation-draft.json",
    OUTPUT_DIR / "head" / "training.csv",
    OUTPUT_DIR / "fine-tune" / "training.csv",
]
missing_outputs = [path.name for path in required_outputs if not path.is_file()]
if missing_outputs:
    raise RuntimeError(f"Training completed without required outputs: {missing_outputs}")
print("Final model, logs, aggregate metrics, and validation evidence are in Google Drive.")

## 12. Remove sensitive temporary data

Run this after training succeeds, or whenever you stop an unsuccessful run.

In [ ]:
import shutil

shutil.rmtree(DATA_ROOT, ignore_errors=True)
print("Temporary dataset and protected manifest removed.")

## 13. Stop the runtime

Confirm the outputs exist in Drive, then select
**Runtime > Disconnect and delete runtime**.